# Whisper + LLM Romanian ASR Experiment

Use this notebook as the main entry point for the project. Edit the configuration cell below, run the preview cell to verify the manifest subset, then run one stage at a time or the full pipeline.

## 1. Imports and Project Setup

In [1]:
from pathlib import Path
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, deep_merge, output_path
from src.data import load_dataset
from src.io_utils import ensure_output_dirs
from run_whisper_transcription import run as run_whisper_transcription
from run_llm_correction import run as run_llm_correction
from evaluate_results import run as run_evaluation

print(f"Project root: {PROJECT_ROOT}")

c:\Users\Sebastian\miniconda3\envs\da\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: D:\whisper_llm


## 2. Edit Experiment Configuration

Set your manifest, audio root, model path, subset size, and Ollama options here. Relative paths are resolved from the project root.

In [2]:
# Paths
manifest_path = "../manifest.csv"
audio_root = "../"
whisper_model_path = "FT_ALL_whisper_large_v2"
output_dir = "outputs"

# Dataset subset
split_to_use = "test"
sources_to_use = ["common_voice", "echo", "fleurs", "voxpopuli", "rodigits"]
max_samples_per_source = None  # Use None for all samples
sampling_strategy = "balanced_per_source"  # "balanced_per_source" or "first_n"
random_seed = 42
resume = True

# Ollama / LLM
ollama_url = "http://localhost:11434/api/generate"
llm_model = "llama3.1:8b"
llm_temperature = 0.0
llm_num_predict = 256
llm_max_retries = 3
llm_timeout_sec = 120

# Text normalization for metrics
normalization_lowercase = True
normalization_remove_punctuation = True
normalization_keep_diacritics = True

# Whisper generation
whisper_language = "ro"
whisper_task = "transcribe"
whisper_max_new_tokens = 256

# Pipeline toggles
run_whisper_stage = False
run_llm_stage = True
run_eval_stage = True

## 3. Build Notebook Config

This cell merges your values with `config.yaml`, resolves paths, and writes `outputs/notebook_config.yaml`. The scripts below use that generated config file.

In [3]:
def resolve_project_path(value):
    if value is None:
        return None
    path = Path(str(value)).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return str(path.resolve())


base_config_path = PROJECT_ROOT / "config.yaml"
base_config = load_config(base_config_path)

notebook_overrides = {
    "manifest_path": resolve_project_path(manifest_path),
    "audio_root": resolve_project_path(audio_root),
    "whisper_model_path": resolve_project_path(whisper_model_path),
    "output_dir": resolve_project_path(output_dir),
    "split_to_use": split_to_use,
    "sources_to_use": sources_to_use,
    "max_samples_per_source": max_samples_per_source,
    "random_seed": random_seed,
    "sampling_strategy": sampling_strategy,
    "resume": resume,
    "ollama_url": ollama_url,
    "llm_model": llm_model,
    "llm_temperature": llm_temperature,
    "llm_num_predict": llm_num_predict,
    "llm_max_retries": llm_max_retries,
    "llm_timeout_sec": llm_timeout_sec,
    "normalization": {
        "lowercase": normalization_lowercase,
        "remove_punctuation": normalization_remove_punctuation,
        "keep_diacritics": normalization_keep_diacritics,
    },
    "whisper": {
        "sampling_rate": 16000,
        "language": whisper_language,
        "task": whisper_task,
        "max_new_tokens": whisper_max_new_tokens,
    },
}

config = deep_merge(base_config, notebook_overrides)
ensure_output_dirs(config["output_dir"])

notebook_config_path = Path(config["output_dir"]) / "notebook_config.yaml"
with notebook_config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print(f"Notebook config written to: {notebook_config_path}")
print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))

Notebook config written to: D:\whisper_llm\outputs\notebook_config.yaml
manifest_path: D:\manifest.csv
audio_root: D:\
whisper_model_path: D:\whisper_llm\FT_ALL_whisper_large_v2
output_dir: D:\whisper_llm\outputs
split_to_use: test
sources_to_use:
- common_voice
- echo
- fleurs
- voxpopuli
- rodigits
max_samples_per_source: null
random_seed: 42
sampling_strategy: balanced_per_source
resume: true
predictions_filename: whisper_predictions.csv
corrections_filename: llm_corrections.csv
ollama_url: http://localhost:11434/api/generate
llm_model: llama3.1:8b
llm_temperature: 0.0
llm_num_predict: 256
llm_max_retries: 3
llm_timeout_sec: 120
normalization:
  lowercase: true
  remove_punctuation: true
  keep_diacritics: true
whisper:
  sampling_rate: 16000
  language: ro
  task: transcribe
  max_new_tokens: 256



## 4. Preview Manifest Subset

Run this before launching Whisper. It checks filtering, audio existence, and sampling.

In [4]:
if config.get("manifest_path") is None:
    raise ValueError("Set manifest_path in the configuration cell first.")

dataset = load_dataset(config)
print(f"Samples after filtering/sampling: {len(dataset)}")
display(dataset.groupby("source").size().reset_index(name="n"))
display(dataset.head())

D:\whisper_llm\src\data.py:31: DtypeWarning: Columns (5,6,9,10,12,13,14,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Samples after filtering/sampling: 23995


,source,n
0,common_voice,3946
1,echo,16294
2,fleurs,883
3,rodigits,1489
4,voxpopuli,1383


,sample_id,audio_path,text,source,split,original_split,speaker_id,gender,age,accent,...,audio_format,recording_id,text_path,script_path,original_path,original_audio_path,arrow_file,row_idx,manifest_file,audio_abs_path
0,9119,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,Domnia sa poate să-şi apere drepturile sale pr...,common_voice,test,test,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,common_voice_test_manifest.csv,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...
1,9120,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,"Avem, însă, și o responsabilitate europeană.",common_voice,test,test,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,common_voice_test_manifest.csv,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...
2,9121,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,Acum am depus două amendamente în plen.,common_voice,test,test,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,common_voice_test_manifest.csv,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...
3,9122,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,Iti mai amintesti?,common_voice,test,test,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,common_voice_test_manifest.csv,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...
4,9123,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,Mica afacere a tatălui meu rămâne mică.,common_voice,test,test,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,common_voice_test_manifest.csv,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...


## 5. Run Selected Pipeline Stages

The same resume behavior as the CLI scripts applies here. If `resume = True`, already processed `sample_id` values are skipped.

In [5]:
if run_whisper_stage:
    predictions_path = run_whisper_transcription(str(notebook_config_path))
    print(f"Whisper predictions: {predictions_path}")
else:
    predictions_path = output_path(config, "predictions", config["predictions_filename"])
    print("Skipped Whisper stage")

if run_llm_stage:
    corrections_path = run_llm_correction(str(notebook_config_path))
    print(f"LLM corrections: {corrections_path}")
else:
    corrections_path = output_path(config, "corrections", config["corrections_filename"])
    print("Skipped LLM stage")

if run_eval_stage:
    global_metrics_path = run_evaluation(str(notebook_config_path))
    print(f"Global metrics: {global_metrics_path}")
else:
    global_metrics_path = output_path(config, "metrics", "metrics_global.json")
    print("Skipped evaluation stage")

Skipped Whisper stage
2026-07-03 21:55:38,499 | INFO | Loaded 23995 predictions; 8098 pending for LLM.


LLM correction: 100%|██████████| 8098/8098 [6:51:20<00:00,  3.05s/it]  

2026-07-04 04:46:59,348 | INFO | LLM corrections saved to D:\whisper_llm\outputs\corrections\llm_corrections.csv


LLM corrections: D:\whisper_llm\outputs\corrections\llm_corrections.csv
2026-07-04 04:49:23,798 | INFO | Per-sample metrics saved to D:\whisper_llm\outputs\metrics\metrics_per_sample.csv
2026-07-04 04:49:23,799 | INFO | Per-source metrics saved to D:\whisper_llm\outputs\metrics\metrics_per_source.csv
2026-07-04 04:49:23,800 | INFO | Global metrics saved to D:\whisper_llm\outputs\metrics\metrics_global.json
2026-07-04 04:49:23,801 | INFO | XGBoost feature table saved to D:\whisper_llm\outputs\metrics\features_for_xgboost.csv
Global metrics: D:\whisper_llm\outputs\metrics\metrics_global.json


## 6. Inspect Outputs

In [6]:
metrics_dir = Path(config["output_dir"]) / "metrics"
per_sample_path = metrics_dir / "metrics_per_sample.csv"
per_source_path = metrics_dir / "metrics_per_source.csv"
features_path = metrics_dir / "features_for_xgboost.csv"

for path in [predictions_path, corrections_path, per_sample_path, per_source_path, features_path, global_metrics_path]:
    print(path, "exists=" + str(Path(path).exists()))

if per_source_path.exists():
    display(pd.read_csv(per_source_path))

if per_sample_path.exists():
    display(pd.read_csv(per_sample_path).head())

D:\whisper_llm\outputs\predictions\whisper_predictions.csv exists=True
D:\whisper_llm\outputs\corrections\llm_corrections.csv exists=True
D:\whisper_llm\outputs\metrics\metrics_per_sample.csv exists=True
D:\whisper_llm\outputs\metrics\metrics_per_source.csv exists=True
D:\whisper_llm\outputs\metrics\features_for_xgboost.csv exists=True
D:\whisper_llm\outputs\metrics\metrics_global.json exists=True


,source,n,wer_before_percent,wer_after_percent,wer_gain_percent,cer_before_percent,cer_after_percent,cer_gain_percent,improved_percent,degraded_percent,unchanged_percent,changed_by_llm_percent
0,common_voice,3946,10.309314,24.393723,-14.084409,2.573174,11.864336,-9.291162,4.510897,51.596553,43.892549,99.974658
1,echo,16294,6.816405,15.026303,-8.209898,2.162717,8.315948,-6.153231,10.580582,55.020253,34.399165,100.000000
2,fleurs,883,9.533654,15.615385,-6.081731,3.341906,8.511145,-5.169239,17.440544,59.343148,23.216308,99.546999
3,rodigits,1489,0.246250,24.227670,-23.981419,0.210231,21.339603,-21.129371,0.000000,63.868368,36.131632,100.000000
4,voxpopuli,1383,10.239197,17.637108,-7.397912,4.805458,8.559159,-3.753701,12.653651,66.088214,21.258134,100.000000


,sample_id,source,split,audio_path,audio_abs_path,reference,whisper_prediction,llm_corrected_prediction,llm_status,llm_model,...,whisper_prediction_norm,llm_corrected_prediction_norm,wer_before,wer_after,cer_before,cer_after,wer_gain,cer_gain,status_wer,status_cer
0,9119,common_voice,test,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...,Domnia sa poate să-şi apere drepturile sale pr...,domnia sa poate să-și apere drepturile sale pr...,"""Dominia sa poate să-și apere drepturile sale ...",ok,llama3.1:8b,...,domnia sa poate să și apere drepturile sale pr...,dominia sa poate să și apere drepturile sale p...,0.111111,0.333333,0.018868,0.132075,-0.222222,-0.113208,degraded,degraded
1,9120,common_voice,test,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...,"Avem, însă, și o responsabilitate europeană.",Avem însă și o responsabilitate europeană.,"„Avem, în schimb, și o responsabilitate europe...",ok,llama3.1:8b,...,avem însă și o responsabilitate europeană,avem în schimb și o responsabilitate europeană,0.000000,0.333333,0.000000,0.146341,-0.333333,-0.146341,degraded,degraded
2,9121,common_voice,test,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...,Acum am depus două amendamente în plen.,Acum am depus două amendamente în plen.,„Acum am depus două amendamente în plen.”,ok,llama3.1:8b,...,acum am depus două amendamente în plen,acum am depus două amendamente în plen,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unchanged,unchanged
3,9122,common_voice,test,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...,Iti mai amintesti?,îți mai amintești,"„Și ți-ai amintit?""",ok,llama3.1:8b,...,îți mai amintești,și ți ai amintit,0.666667,1.333333,0.176471,0.411765,-0.666667,-0.235294,degraded,degraded
4,9123,common_voice,test,data/cv-corpus-25.0-2026-03-09/ro/clips/common...,D:\data\cv-corpus-25.0-2026-03-09\ro\clips\com...,Mica afacere a tatălui meu rămâne mică.,Mica afacerea tatălui meu rămâne mică.,"""Mica afacere a tatălui meu rămâne mică.""",ok,llama3.1:8b,...,mica afacerea tatălui meu rămâne mică,mica afacere a tatălui meu rămâne mică,0.285714,0.000000,0.026316,0.000000,0.285714,0.026316,improved,improved
